### Tokenizer 

This section uses the SentencePiece tokenizer for making an extended tokenizer for the respective languages that's not in the current supported language list of M2M100.

In [1]:
import sentencepiece as spm
import os
# file,language_code 
# os.makedirs('vocabs')
dataset_lang_corpora = [
    ("dataset/cleaned-txt/bicol_cleaned.sentences.txt","bcl"),
    ("dataset/cleaned-txt/chavacano_cleaned.sentences.txt","chv"),
    ("dataset/cleaned-txt/bantoanon_cleaned.sentences.txt","btn"),
    #("dataset/cleaned-txt/cebuano_cleaned.sentences.txt","ceb"),
    #("dataset/cleaned-txt/english_cleaned.sentences.txt","en"),
    #("dataset/cleaned-txt/filipino-tl_cleaned.sentences.txt","tl"),
    #("dataset/cleaned-txt/ilokano_cleaned.sentences.txt","ilk"),
    ("dataset/cleaned-txt/ilonggo_cleaned.sentences.txt","ilg"),
    ("dataset/cleaned-txt/ivatan_cleaned.sentences.txt","ivt"),
    ("dataset/cleaned-txt/kinaray-a_cleaned.sentences.txt","kry"),
    ("dataset/cleaned-txt/manobo_cleaned.sentences.txt","mbo"),
    ("dataset/cleaned-txt/masbatenyo_cleaned.sentences.txt","mbyo"),
    ("dataset/cleaned-txt/pampanga_cleaned.sentences.txt","pam"),
    ("dataset/cleaned-txt/pangasinan_cleaned.sentences.txt","pgn"),
    ("dataset/cleaned-txt/romblomanon_cleaned.sentences.txt","rblo"),
    ("dataset/cleaned-txt/sambal_cleaned.sentences.txt","sbl"),
    #("dataset/cleaned-txt/spanish_cleaned.sentences.txt","spa"),
    ("dataset/cleaned-txt/tausug_cleaned.sentences.txt","tsg"),
    ("dataset/cleaned-txt/waray_cleaned.sentences.txt","wry"),
    ("dataset/cleaned-txt/yakan_cleaned.sentences.txt","ykn"),
    ("dataset/cleaned-txt/yami_cleaned.sentences.txt","ymi")

]


In [2]:
def train_tokenizer(input_file,lang_code,vocab_size=3000,char_coverage=0.9995,model_type="bpe"):
    model_prefix = "vocabs/"+lang_code+"-spe"
    print(model_prefix)
    spm.SentencePieceTrainer.train(input=input_file,
                                   model_prefix=model_prefix,
                                   character_coverage=char_coverage,
                                   vocab_size=vocab_size,
                                   model_type=model_type,                                   
                                   )

#for i in range(0,len(dataset_lang_corpora)):
 #   train_tokenizer(dataset_lang_corpora[i][0],dataset_lang_corpora[i][1])

### Extending M2M100 1.2B Tokenizer


In [5]:
from transformers import M2M100Tokenizer, M2M100ForConditionalGeneration
import json
model_name = "facebook/m2m100_1.2B"
model = M2M100ForConditionalGeneration.from_pretrained(model_name)
main_tokenizer = M2M100Tokenizer.from_pretrained(model_name)
print("Original Length:"+str(len(main_tokenizer)))
# sentence piece generates a .vocab file instead of a .json format
def spm_to_json_vocab(spm_model_path, json_vocab_path):
    sp = spm.SentencePieceProcessor()
    sp.load(spm_model_path)

    vocab = { sp.id_to_piece(i): i for i in range(sp.get_piece_size()) }

    with open(json_vocab_path, "w") as f:
        json.dump(vocab, f, ensure_ascii=False, indent=2)


def extend_model_with_lang_code(lang_code):
    spm_to_json_vocab("vocabs/"+lang_code+"-spe.model","vocabs/"+lang_code+"-spe.json")
    new_tokenizer = M2M100Tokenizer(spm_file="vocabs/"+lang_code+"-spe.model",vocab_file="vocabs/"+lang_code+"-spe.json")
    total_tokens_added = main_tokenizer.add_tokens(list(new_tokenizer.get_vocab().keys()))
    print(f"{lang_code}: added {total_tokens_added} tokens")



for i in range(0,len(dataset_lang_corpora)):
    extend_model_with_lang_code(dataset_lang_corpora[i][1])

model.resize_token_embeddings(len(main_tokenizer),mean_resizing=False)

#os.makedirs("tokenizers")
print(len(main_tokenizer))
main_tokenizer.save_pretrained("tokenizers/extended_m2m100tokenizer")
model.save_pretrained("models/extended_lang_m2m100_1.2B")



Original Length:128104
bcl: added 1675 tokens
chv: added 1337 tokens
btn: added 1529 tokens
ilg: added 1496 tokens
ivt: added 1844 tokens
kry: added 1077 tokens
mbo: added 1746 tokens
mbyo: added 1013 tokens
pam: added 1290 tokens
pgn: added 1405 tokens
rblo: added 958 tokens
sbl: added 1466 tokens
tsg: added 1605 tokens
wry: added 955 tokens
ykn: added 1477 tokens
ymi: added 1894 tokens
150871


/usr/local/lib/python3.12/site-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


In [ ]:
from transformers import M2M100Tokenizer

tokenizer = M2M100Tokenizer.from_pretrained("./models/extended_lang_m2m100_1.2B/checkpoint-17511")
print(len(tokenizer))

150871
